# Qualidade de Dados

In [0]:
CATALOG = "workspace"
from pyspark.sql import functions as F

> Avaliação de completude

Calcula o percentual de valores nulos em atributos relevantes da base de títulos, permitindo identificar possíveis problemas de preenchimento.

In [0]:
bronze_basics = spark.table(f"{CATALOG}.bronze.title_basics")
total = bronze_basics.count()

completude = bronze_basics.select(
    *[
        (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(f"{c}_pct_nulo")
        for c in ["startYear", "endYear", "runtimeMinutes", "genres"]
    ]
)
display(completude)

startYear_pct_nulo,endYear_pct_nulo,runtimeMinutes_pct_nulo,genres_pct_nulo
11.574355125784615,98.73514377128916,62.546446133953516,4.2409665024532925


> Avaliação de unicidade

Verifica a existência de identificadores duplicados na camada Bronze e confirma sua remoção após o tratamento realizado na camada Silver.

In [0]:
dup_tconst = (
    bronze_basics.groupBy("tconst").count().filter(F.col("count") > 1).count()
)
print(f"tconst duplicados em title_basics (bronze): {dup_tconst}")

dup_silver = (
    spark.table(f"{CATALOG}.silver.title_basics")
    .groupBy("tconst").count().filter(F.col("count") > 1).count()
)
print(f"tconst duplicados em title_basics (silver, após dropDuplicates): {dup_silver}")

tconst duplicados em title_basics (bronze): 0
tconst duplicados em title_basics (silver, após dropDuplicates): 0


In [0]:
silver_ratings_raw = (
    spark.table(f"{CATALOG}.bronze.title_ratings")
    .withColumn(
        "averageRating",
        F.expr("try_cast(averageRating as double)")
    )
)

fora_do_dominio = silver_ratings_raw.filter(
    (F.col("averageRating") < 0) | (F.col("averageRating") > 10)
).count()
print(f"Notas fora do intervalo [0, 10]: {fora_do_dominio}")

runtime_negativo = (
    spark.table(f"{CATALOG}.silver.title_basics")
    .filter(F.col("runtimeMinutes") < 0)
    .count()
)

print(f"runtimeMinutes negativo: {runtime_negativo}")

Notas fora do intervalo [0, 10]: 0
runtimeMinutes negativo: 0


> Distribuição dos tipos de produção

Exibe a quantidade de registros por tipo de título presente na base IMDb.

In [0]:
display(
    spark.table(f"{CATALOG}.bronze.title_basics")
    .groupBy("titleType").count().orderBy(F.desc("count"))
)

titleType,count
tvEpisode,9911000
short,1159572
movie,757802
video,331573
tvSeries,305838
tvMovie,156333
tvMiniSeries,73205
tvSpecial,60499
videoGame,50372
tvShort,11072


In [0]:
silver_basics = spark.table(f"{CATALOG}.silver.title_basics")

display(
    silver_basics.filter(F.col("runtimeMinutes").isNotNull())
    .select(
        F.min("runtimeMinutes").alias("min_runtime"),
        F.expr("percentile(runtimeMinutes, 0.5)").alias("mediana_runtime"),
        F.max("runtimeMinutes").alias("max_runtime"),
    )
)

min_runtime,mediana_runtime,max_runtime
1,42.0,3692080


In [0]:

display(
    spark.table(f"{CATALOG}.silver.title_ratings")
    .select(
        F.min("numVotes").alias("min_votos"),
        F.expr("percentile(numVotes, 0.5)").alias("mediana_votos"),
        F.expr("percentile(numVotes, 0.99)").alias("p99_votos"),
        F.max("numVotes").alias("max_votos"),
    )
)

min_votos,mediana_votos,p99_votos,max_votos
5,26.0,10862.0,3243065
